# Bulgarian ASR data collection

Run the setup cell once, then run dataset cells one by one.
Each dataset cell downloads archives into `data/row_data/_archives/` and extracts into its own folder inside `data/row_data/`.

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import gzip
import shutil
import subprocess
import tarfile
import urllib.request
import zipfile

RAW_DIR = Path("data/row_data")
ARCHIVE_DIR = RAW_DIR / "_archives"
RAW_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)


def _safe_name_from_url(url: str, fallback: str = "downloaded_file") -> str:
    path = urlparse(url).path
    name = Path(path).name
    return name or fallback


def _download_with_fallback(url: str, dst: Path) -> None:
    errors = []

    try:
        urllib.request.urlretrieve(url, dst)
        return
    except Exception as exc:
        errors.append(f"urllib failed: {exc}")

    try:
        subprocess.run(
            ["curl", "-L", "--fail", "-o", str(dst), url],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        return
    except Exception as exc:
        errors.append(f"curl failed: {exc}")

    try:
        subprocess.run(
            ["wget", "-O", str(dst), url],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        return
    except Exception as exc:
        errors.append(f"wget failed: {exc}")

    raise RuntimeError(" | ".join(errors))


def _extract_archive(archive_path: Path, out_dir: Path) -> bool:
    out_dir.mkdir(parents=True, exist_ok=True)
    name = archive_path.name.lower()

    if name.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(out_dir)
        return True

    if (
        name.endswith(".tar")
        or name.endswith(".tar.gz")
        or name.endswith(".tgz")
        or name.endswith(".tar.bz2")
        or name.endswith(".tbz2")
        or name.endswith(".tar.xz")
        or name.endswith(".txz")
    ):
        with tarfile.open(archive_path, "r:*") as tf:
            tf.extractall(out_dir)
        return True

    if name.endswith(".gz") and not name.endswith(".tar.gz"):
        output_file = out_dir / archive_path.with_suffix("").name
        with gzip.open(archive_path, "rb") as src, open(output_file, "wb") as dst:
            shutil.copyfileobj(src, dst)
        return True

    return False


def _is_invalid_local_file(path: Path) -> bool:
    return (not path.exists()) or path.stat().st_size == 0


def fetch_dataset(
    dataset_id: str,
    urls: list[str],
    archive_name: str | None = None,
    force_download: bool = False,
    clean_extract_dir: bool = False,
):
    if not urls:
        print(f"[{dataset_id}] No URLs provided. Add at least one URL and rerun.")
        return None

    dataset_dir = RAW_DIR / dataset_id
    if clean_extract_dir and dataset_dir.exists():
        shutil.rmtree(dataset_dir)
    dataset_dir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failed = []

    for idx, url in enumerate(urls, start=1):
        filename = archive_name if (archive_name and len(urls) == 1) else _safe_name_from_url(url, fallback=f"{dataset_id}_{idx}")
        archive_path = ARCHIVE_DIR / filename

        try:
            if force_download and archive_path.exists():
                archive_path.unlink()

            if archive_path.exists() and archive_path.stat().st_size == 0:
                print(f"[{dataset_id}] Removing empty cached file: {archive_path}")
                archive_path.unlink()

            if archive_path.exists():
                print(f"[{dataset_id}] Archive already exists: {archive_path}")
            else:
                print(f"[{dataset_id}] Downloading: {url}")
                _download_with_fallback(url, archive_path)
                if _is_invalid_local_file(archive_path):
                    raise RuntimeError("Downloaded file is empty or missing")
                print(f"[{dataset_id}] Saved to: {archive_path}")

            extracted = _extract_archive(archive_path, dataset_dir)
            if extracted:
                print(f"[{dataset_id}] Extracted: {archive_path.name}")
            else:
                shutil.copy2(archive_path, dataset_dir / archive_path.name)
                print(f"[{dataset_id}] Not an archive, copied as file: {archive_path.name}")

            success_count += 1

        except Exception as exc:
            failed.append((url, str(exc)))
            # Remove failed cache file so next URL cannot reuse a bad artifact.
            if archive_path.exists():
                archive_path.unlink()
            print(f"[{dataset_id}] Failed URL: {url}")
            print(f"[{dataset_id}] Error: {exc}")

    if success_count == 0:
        last_error = failed[-1][1] if failed else "unknown error"
        raise RuntimeError(f"[{dataset_id}] All URLs failed. Last error: {last_error}")

    print(f"[{dataset_id}] Completed. Successful items: {success_count}, failed items: {len(failed)}")
    return {
        "dataset_dir": str(dataset_dir),
        "successful_items": success_count,
        "failed_items": failed,
    }

In [5]:
# 1) Mozilla Common Voice (BG)
# Verified source mirror (HF):
# https://huggingface.co/datasets/IliyanGochev/common_voice_13_0_bg_pseudo_labelled
URLS = [
    "https://huggingface.co/datasets/IliyanGochev/common_voice_13_0_bg_pseudo_labelled/resolve/main/bg/train-00000-of-00001.parquet",
    "https://huggingface.co/datasets/IliyanGochev/common_voice_13_0_bg_pseudo_labelled/resolve/main/bg/validation-00000-of-00001.parquet",
    "https://huggingface.co/datasets/IliyanGochev/common_voice_13_0_bg_pseudo_labelled/resolve/main/bg/test-00000-of-00001.parquet",
]

fetch_dataset(
    dataset_id="mozilla_common_voice_bg",
    urls=URLS,
    force_download=False,
    clean_extract_dir=False,
)

[mozilla_common_voice_bg] Downloading: https://huggingface.co/datasets/IliyanGochev/common_voice_13_0_bg_pseudo_labelled/resolve/main/bg/train-00000-of-00001.parquet
[mozilla_common_voice_bg] Saved to: data/row_data/_archives/train-00000-of-00001.parquet
[mozilla_common_voice_bg] Not an archive, copied as file: train-00000-of-00001.parquet
[mozilla_common_voice_bg] Downloading: https://huggingface.co/datasets/IliyanGochev/common_voice_13_0_bg_pseudo_labelled/resolve/main/bg/validation-00000-of-00001.parquet
[mozilla_common_voice_bg] Saved to: data/row_data/_archives/validation-00000-of-00001.parquet
[mozilla_common_voice_bg] Not an archive, copied as file: validation-00000-of-00001.parquet
[mozilla_common_voice_bg] Downloading: https://huggingface.co/datasets/IliyanGochev/common_voice_13_0_bg_pseudo_labelled/resolve/main/bg/test-00000-of-00001.parquet
[mozilla_common_voice_bg] Saved to: data/row_data/_archives/test-00000-of-00001.parquet
[mozilla_common_voice_bg] Not an archive, copied

{'dataset_dir': 'data/row_data/mozilla_common_voice_bg',
 'successful_items': 3,
 'failed_items': []}

In [6]:
# 2) CC0 Bulgarian Speech Dataset (Shunya Labs)
# Verified source:
# https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset
URLS = [
    "https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/train-00000-of-00005.parquet",
    "https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/train-00001-of-00005.parquet",
    "https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/train-00002-of-00005.parquet",
    "https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/train-00003-of-00005.parquet",
    "https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/train-00004-of-00005.parquet",
    "https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/validation-00000-of-00001.parquet",
    "https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/test-00000-of-00001.parquet",
]

fetch_dataset(
    dataset_id="cc0_bulgarian_speech_shunya",
    urls=URLS,
    force_download=False,
    clean_extract_dir=False,
)

[cc0_bulgarian_speech_shunya] Downloading: https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/train-00000-of-00005.parquet
[cc0_bulgarian_speech_shunya] Saved to: data/row_data/_archives/train-00000-of-00005.parquet
[cc0_bulgarian_speech_shunya] Not an archive, copied as file: train-00000-of-00005.parquet
[cc0_bulgarian_speech_shunya] Downloading: https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/train-00001-of-00005.parquet
[cc0_bulgarian_speech_shunya] Saved to: data/row_data/_archives/train-00001-of-00005.parquet
[cc0_bulgarian_speech_shunya] Not an archive, copied as file: train-00001-of-00005.parquet
[cc0_bulgarian_speech_shunya] Downloading: https://huggingface.co/datasets/shunyalabs/bulgarian-speech-dataset/resolve/main/data/train-00002-of-00005.parquet
[cc0_bulgarian_speech_shunya] Saved to: data/row_data/_archives/train-00002-of-00005.parquet
[cc0_bulgarian_speech_shunya] Not an archive, copied as file: tra

{'dataset_dir': 'data/row_data/cc0_bulgarian_speech_shunya',
 'successful_items': 7,
 'failed_items': []}

In [13]:
# 3) BG-PARLAMA (Bulgarian Parliament Corpus)
# This corpus may be access-restricted for academic use.
# Option A: put working direct URLs in URLS.
# Option B: if you received an archive manually, set LOCAL_ARCHIVE_PATH.

URLS = [
    # "http://.../BG-PARLAMA.tar.gz",
]

LOCAL_ARCHIVE_PATH = ""  # e.g. "data/row_data/_archives/BG-PARLAMA.tar.gz"

if URLS:
    try:
        fetch_dataset(
            dataset_id="bg_parlama",
            urls=URLS,
            archive_name="bg_parlama.tar.gz",
            force_download=False,
            clean_extract_dir=False,
        )
    except RuntimeError as e:
        print(e)
        print("\nBG-PARLAMA direct download is unavailable. Use manual access/request and set LOCAL_ARCHIVE_PATH.")
elif LOCAL_ARCHIVE_PATH:
    local_path = Path(LOCAL_ARCHIVE_PATH)
    if not local_path.exists():
        print(f"Local archive not found: {local_path}")
    else:
        dataset_dir = RAW_DIR / "bg_parlama"
        dataset_dir.mkdir(parents=True, exist_ok=True)
        extracted = _extract_archive(local_path, dataset_dir)
        if extracted:
            print(f"[bg_parlama] Extracted local archive: {local_path}")
        else:
            shutil.copy2(local_path, dataset_dir / local_path.name)
            print(f"[bg_parlama] Local file copied (not archive): {local_path}")
else:
    print("BG-PARLAMA likely requires academic access request.")
    print("Set URLS (if you got direct links) or LOCAL_ARCHIVE_PATH (if you got the file manually).")

BG-PARLAMA likely requires academic access request.
Set URLS (if you got direct links) or LOCAL_ARCHIVE_PATH (if you got the file manually).


In [ ]:
# # 4) BulPhonC
# URLS = [
#     # "https://.../bulphonc.zip",
# ]

# fetch_dataset(
#     dataset_id="bulphonc",
#     urls=URLS,
#     archive_name="bulphonc.zip",
#     force_download=False,
#     clean_extract_dir=False,
# )

In [3]:
# 5) VoxForge Bulgarian
# Source index:
# http://www.repository.voxforge1.org/downloads/bg/Trunk/Audio/Main/16kHz_16bit/
BASE = "http://www.repository.voxforge1.org/downloads/bg/Trunk/Audio/Main/16kHz_16bit"
URLS = [
    f"{BASE}/Adi-20110313-pbk.tgz",
    f"{BASE}/AlexGotev-20111017-kli.tgz",
    f"{BASE}/BGGeorgi-20110801-aag.tgz",
    f"{BASE}/FF-20131202-lxj.tgz",
    f"{BASE}/FF-20131202-rqu.tgz",
    f"{BASE}/FF-20131202-xng.tgz",
    f"{BASE}/Garo02-20140324-lhl.tgz",
    f"{BASE}/Grountex-20130211-kjy.tgz",
    f"{BASE}/Ivo-20110625-ygd.tgz",
    f"{BASE}/Vlad_Cepesh-20120506-upq.tgz",
]

fetch_dataset(
    dataset_id="voxforge_bulgarian",
    urls=URLS,
    force_download=False,
    clean_extract_dir=False,
)

[voxforge_bulgarian] Downloading: http://www.repository.voxforge1.org/downloads/bg/Trunk/Audio/Main/16kHz_16bit/Adi-20110313-pbk.tgz
[voxforge_bulgarian] Saved to: data/row_data/_archives/Adi-20110313-pbk.tgz
[voxforge_bulgarian] Extracted: Adi-20110313-pbk.tgz
[voxforge_bulgarian] Downloading: http://www.repository.voxforge1.org/downloads/bg/Trunk/Audio/Main/16kHz_16bit/AlexGotev-20111017-kli.tgz
[voxforge_bulgarian] Saved to: data/row_data/_archives/AlexGotev-20111017-kli.tgz
[voxforge_bulgarian] Extracted: AlexGotev-20111017-kli.tgz
[voxforge_bulgarian] Downloading: http://www.repository.voxforge1.org/downloads/bg/Trunk/Audio/Main/16kHz_16bit/BGGeorgi-20110801-aag.tgz
[voxforge_bulgarian] Saved to: data/row_data/_archives/BGGeorgi-20110801-aag.tgz
[voxforge_bulgarian] Extracted: BGGeorgi-20110801-aag.tgz
[voxforge_bulgarian] Downloading: http://www.repository.voxforge1.org/downloads/bg/Trunk/Audio/Main/16kHz_16bit/FF-20131202-lxj.tgz
[voxforge_bulgarian] Saved to: data/row_data/_arc

{'dataset_dir': 'data/row_data/voxforge_bulgarian',
 'successful_items': 10,
 'failed_items': []}

In [ ]:
# # 6) Bulgarian General Conversation Speech
# URLS = [
#     # "https://.../bulgarian_general_conversation_speech.zip",
# ]

# fetch_dataset(
#     dataset_id="bulgarian_general_conversation_speech",
#     urls=URLS,
#     archive_name="bulgarian_general_conversation_speech.zip",
#     force_download=False,
#     clean_extract_dir=False,
# )

In [ ]:
# # 7) Bulgarian Parliament (extended corpora)
# URLS = [
#     # "https://.../bulgarian_parliament_extended.tar.gz",
# ]

# fetch_dataset(
#     dataset_id="bulgarian_parliament_extended",
#     urls=URLS,
#     archive_name="bulgarian_parliament_extended.tar.gz",
#     force_download=False,
#     clean_extract_dir=False,
# )

In [7]:
from collections import defaultdict
from io import BytesIO
from statistics import median

AUDIO_EXTENSIONS = {
    ".wav", ".flac", ".mp3", ".m4a", ".aac", ".ogg", ".opus", ".wma", ".aiff", ".aif", ".alac"
}
PARQUET_DATASET_DIRS = {
    "mozilla_common_voice_bg",
    "cc0_bulgarian_speech_shunya",
}


def _duration_from_wav_bytes(payload: bytes) -> float:
    import contextlib
    import wave

    with contextlib.closing(wave.open(BytesIO(payload), "rb")) as wf:
        frames = wf.getnframes()
        rate = wf.getframerate()
        if not rate:
            raise ValueError("Zero sample rate in embedded WAV payload")
        return frames / float(rate)


def _duration_from_audio_bytes(payload: bytes, filename: str | None = None) -> float:
    try:
        import soundfile as sf

        with sf.SoundFile(BytesIO(payload)) as audio:
            if not audio.samplerate:
                raise ValueError("Zero sample rate in embedded audio payload")
            return len(audio) / float(audio.samplerate)
    except Exception:
        pass

    try:
        return _duration_from_wav_bytes(payload)
    except Exception as exc:
        raise RuntimeError("Cannot read embedded audio duration") from exc


def _duration_from_audio_file(path: Path) -> float:
    suffix = path.suffix.lower()

    if suffix == ".wav":
        import contextlib
        import wave

        with contextlib.closing(wave.open(str(path), "rb")) as wf:
            frames = wf.getnframes()
            rate = wf.getframerate()
            if not rate:
                raise ValueError(f"Zero sample rate: {path}")
            return frames / float(rate)

    try:
        import soundfile as sf

        info = sf.info(str(path))
        if info.frames is None or info.samplerate is None:
            raise ValueError(f"Incomplete audio info: {path}")
        if not info.samplerate:
            raise ValueError(f"Zero sample rate: {path}")
        return info.frames / float(info.samplerate)
    except Exception:
        pass

    try:
        from mutagen import File as MutagenFile

        audio = MutagenFile(str(path))
        if audio is None or not getattr(audio, "info", None) or not getattr(audio.info, "length", None):
            raise ValueError(f"Unsupported audio file: {path}")
        return float(audio.info.length)
    except Exception as exc:
        raise RuntimeError(f"Cannot read duration for {path}") from exc


def _collect_disk_audio(root_dir: Path) -> list[tuple[str, Path, float]]:
    items = []
    for path in root_dir.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
            continue
        duration = _duration_from_audio_file(path)
        items.append(("file", path, duration))
    return items


def _collect_parquet_audio(root_dir: Path) -> list[tuple[str, Path, float]]:
    try:
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise RuntimeError("pyarrow is required to read parquet audio datasets") from exc

    items = []
    for dataset_dir in PARQUET_DATASET_DIRS:
        dataset_path = root_dir / dataset_dir
        if not dataset_path.exists():
            continue

        for parquet_path in sorted(dataset_path.rglob("*.parquet")):
            table = pq.read_table(parquet_path, columns=["audio"])
            audio_column = table.column("audio")
            for row_index in range(len(audio_column)):
                audio_value = audio_column[row_index].as_py()
                if not isinstance(audio_value, dict) or "bytes" not in audio_value:
                    continue
                payload = audio_value["bytes"]
                if not payload:
                    continue
                filename = audio_value.get("path")
                duration = _duration_from_audio_bytes(payload, filename=filename)
                item_path = parquet_path.with_name(f"{parquet_path.stem}::row-{row_index}")
                items.append(("parquet", item_path, duration))
    return items


all_audio_items = _collect_disk_audio(RAW_DIR) + _collect_parquet_audio(RAW_DIR)

if not all_audio_items:
    print(f"No audio files found under {RAW_DIR}")
else:
    durations = [duration for _, _, duration in all_audio_items]
    shortest_idx = min(range(len(durations)), key=durations.__getitem__)
    longest_idx = max(range(len(durations)), key=durations.__getitem__)

    total_seconds = sum(durations)
    median_seconds = median(durations)
    source_counts = defaultdict(int)
    for source_type, _, _ in all_audio_items:
        source_counts[source_type] += 1

    print(f"Audio items found: {len(durations)}")
    print(
        "By source: "
        + ", ".join(f"{source_type}={count}" for source_type, count in sorted(source_counts.items()))
    )
    print(f"Total duration: {total_seconds:.2f} s ({total_seconds / 3600:.2f} h)")
    print(f"Shortest item: {all_audio_items[shortest_idx][1]} -> {durations[shortest_idx]:.2f} s")
    print(f"Longest item: {all_audio_items[longest_idx][1]} -> {durations[longest_idx]:.2f} s")
    print(f"Median duration: {median_seconds:.2f} s")


Audio items found: 16100
By source: file=100, parquet=16000
Total duration: 105977.68 s (29.44 h)
Shortest item: data/row_data/mozilla_common_voice_bg/train-00000-of-00001::row-2259 -> 2.17 s
Longest item: data/row_data/cc0_bulgarian_speech_shunya/train-00004-of-00005::row-401 -> 60.60 s
Median duration: 5.66 s
